# 📓 Course 1: Neural Networks from Scratch using NumPy
This notebook implements a simple Multi-Layer Perceptron (MLP) from scratch using **NumPy** to demonstrate the foundational concepts of Course 1 of the Deep Learning Specialization:
- Weight and bias initialization
- Activation functions (ReLU, Sigmoid)
- Forward propagation
- Cost calculation
- Backward propagation
- Gradient descent optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

## 1. Activation Functions & Derivatives
We implement Sigmoid and ReLU activations alongside their gradients.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_backward(da, z):
    s = sigmoid(z)
    return da * s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_backward(da, z):
    dz = np.array(da, copy=True)
    dz[z <= 0] = 0
    return dz

## 2. MLP Class Implementation

In [ ]:
class ScratchNeuralNetwork:
    def __init__(self, layer_dims):
        self.parameters = {}
        self.layer_dims = layer_dims
        self.L = len(layer_dims) - 1
        
        # Initialize parameters (He initialization for ReLU, Xavier for Sigmoid)
        for l in range(1, self.L + 1):
            self.parameters[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2 / layer_dims[l-1])
            self.parameters[f'b{l}'] = np.zeros((layer_dims[l], 1))
            
    def forward(self, X):
        caches = {}
        a = X
        
        for l in range(1, self.L):
            a_prev = a
            z = np.dot(self.parameters[f'W{l}'], a_prev) + self.parameters[f'b{l}']
            a = relu(z)
            caches[f'Z{l}'] = z
            caches[f'A{l}'] = a
            
        # Output layer uses sigmoid for binary classification
        z = np.dot(self.parameters[f'W{self.L}'], a) + self.parameters[f'b{self.L}']
        al = sigmoid(z)
        caches[f'Z{self.L}'] = z
        caches[f'A{self.L}'] = al
        
        return al, caches
        
    def compute_cost(self, AL, Y):
        m = Y.shape[1]
        cost = - (1 / m) * np.sum(Y * np.log(AL + 1e-15) + (1 - Y) * np.log(1 - AL + 1e-15))
        return np.squeeze(cost)
        
    def backward(self, X, Y, AL, caches):
        grads = {}
        m = Y.shape[1]
        
        # Output layer gradient
        dAL = - (np.divide(Y, AL + 1e-15) - np.divide(1 - Y, 1 - AL + 1e-15))
        
        dZ = sigmoid_backward(dAL, caches[f'Z{self.L}'])
        A_prev = caches[f'A{self.L-1}']
        grads[f'dW{self.L}'] = (1 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{self.L}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
        grads[f'dA{self.L-1}'] = np.dot(self.parameters[f'W{self.L}'].T, dZ)
        
        # Hidden layers
        for l in reversed(range(1, self.L)):
            dZ = relu_backward(grads[f'dA{l}'], caches[f'Z{l}'])
            A_prev = X if l == 1 else caches[f'A{l-1}']
            grads[f'dW{l}'] = (1 / m) * np.dot(dZ, A_prev.T)
            grads[f'db{l}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
            if l > 1:
                grads[f'dA{l-1}'] = np.dot(self.parameters[f'W{l}'].T, dZ)
                
        return grads
        
    def update_parameters(self, grads, lr):
        for l in range(1, self.L + 1):
            self.parameters[f'W{l}'] -= lr * grads[f'dW{l}']
            self.parameters[f'b{l}'] -= lr * grads[f'db{l}']

## 3. Training on Synthetic XOR Pattern

In [ ]:
# X: 2 features, 4 samples
X = np.array([[0, 0, 1, 1], [0, 1, 0, 1]])
Y = np.array([[0, 1, 1, 0]]) # XOR outputs

nn = ScratchNeuralNetwork(layer_dims=[2, 4, 1])
costs = []

for epoch in range(1000):
    al, caches = nn.forward(X)
    cost = nn.compute_cost(al, Y)
    grads = nn.backward(X, Y, al, caches)
    nn.update_parameters(grads, lr=0.1)
    
    if epoch % 100 == 0:
        costs.append(cost)
        print(f"Epoch {epoch}: Cost = {cost:.6f}")

plt.plot(costs)
plt.title("Scratch NN Training Cost Curve")
plt.xlabel("Iterations (x100)")
plt.ylabel("Binary Crossentropy Cost")
plt.show()